# OcupaPerú — Fase 2 CRISP-DM: Comprensión de los datos

Este notebook **explora los 11 archivos crudos** de MINCETUR *tal como vienen*, sin modificar
nada. Su único objetivo es **entender** cómo son los datos y **detectar los problemas** que la
Fase 3 (Preparación) tendrá que resolver.

> **Importante:** este notebook NO limpia ni transforma. Solo carga, describe y grafica.
> La limpieza ocurre en el notebook de la Fase 3.

**Fuente de los datos:** MINCETUR — Indicadores de Ocupabilidad
([datosabiertos.gob.pe](https://www.datosabiertos.gob.pe/dataset/indicadores-de-ocupabilidad-ministerio-de-comercio-exterior-y-turismo-mincetur))

**Qué hace este notebook:**
1. Carga los 11 CSV crudos (2015 → 2025-S1)
2. Muestra su estructura: filas, columnas, tipos
3. Detecta los **dos esquemas de columnas incompatibles**
4. Encuentra nulos, duplicados y valores imposibles
5. Visualiza la distribución del TNOH y los arribos
6. Resume los problemas que la Fase 3 deberá resolver

## 0 — Configuración

In [ ]:
# Colab: los CSV deben estar en /content
%cd /content
print('📂 Sube los 11 archivos Indicadores_ocupabilidad_*.csv a /content')
print('   (panel izquierdo 📁 → Subir al almacenamiento de sesión) y vuelve a ejecutar.')

import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (9, 4)

archivos = sorted(glob.glob('Indicadores_ocupabilidad_*.csv'))
print(f'Archivos encontrados: {len(archivos)}')
for a in archivos:
    print('  ', a)
if len(archivos) != 11:
    raise FileNotFoundError('Se esperaban 11 archivos Indicadores_ocupabilidad_*.csv. '
                            + ('Súbelos a /content.' if True else 'Cópialos a la misma carpeta de este notebook.'))


---
## 1 — Carga de los archivos crudos

Cargamos cada archivo **sin tocarlo** y anotamos cuántas filas y columnas trae. Fíjate que no
todos tienen el mismo número de columnas: esa es la primera señal de un problema.

In [ ]:
inventario = []
crudos = {}

for f in archivos:
    d = pd.read_csv(f, sep=';', encoding='latin-1', dtype=str)
    crudos[f] = d
    inventario.append({'archivo': f, 'filas': d.shape[0], 'columnas': d.shape[1]})

inv = pd.DataFrame(inventario)
print('Total de filas crudas:', inv.filas.sum())
inv

---
## 2 — El problema de los dos esquemas (schema drift)

MINCETUR **cambió los nombres de todas las columnas en 2024**. Comparemos las columnas de un
archivo viejo (2015) contra uno nuevo (2024) para verlo con nuestros propios ojos.

In [ ]:
viejo = crudos.get('Indicadores_ocupabilidad_2015.csv')
nuevo = crudos.get('Indicadores_ocupabilidad_2024.csv')

print('=== Columnas del esquema VIEJO (2015) ===')
for c in viejo.columns: print('  ', c)
print(f'\nTotal: {viejo.shape[1]} columnas')

print('\n=== Columnas del esquema NUEVO (2024) ===')
for c in nuevo.columns: print('  ', c)
print(f'\nTotal: {nuevo.shape[1]} columnas')

**Lo que se observa:**
- El año pasa de `AÑO` (con tilde) a `ANIO`.
- La ocupación pasa de `TNOH EN EL MES(%)` a `PORCENTAJE_TNOH`.
- El esquema nuevo agrega una columna `FECHA_CORTE`.
- Los nombres viejos usan tildes y mayúsculas con espacios; los nuevos usan snake_case.

👉 Un `pd.concat()` directo **no daría error**, pero produciría columnas duplicadas y medio
vacías. Por eso la Fase 3 primero **renombra** todo a un esquema común antes de unir.

---
## 3 — El separador decimal cambia entre años

Otro detalle silencioso: los archivos de 2015 a 2024 usan **coma** como separador decimal
(ej. `25,4`) y el de 2025 usa **punto** (`25.4`). Si no se unifica, los números no se pueden
leer correctamente.

In [ ]:
# Buscamos la columna de TNOH en cada esquema y mostramos un valor de ejemplo
def muestra_tnoh(d):
    for col in ['TNOH EN EL MES(%)', 'PORCENTAJE_TNOH']:
        if col in d.columns:
            vals = d[col].dropna().astype(str)
            return col, vals.iloc[0] if len(vals) else 'N/A'
    return None, None

for año in ['2015', '2024', '2025_1er_semestre']:
    nombre = f'Indicadores_ocupabilidad_{año}.csv'
    if nombre in crudos:
        col, val = muestra_tnoh(crudos[nombre])
        tiene_coma = ',' in str(val)
        print(f'{año:20} columna={col:20} ejemplo={val:10} '
              f'{"← usa COMA" if tiene_coma else "← usa PUNTO"}')

---
## 4 — Filas agregadas mezcladas con el detalle

Los archivos mezclan filas de **detalle** (un departamento concreto) con filas de **totales**
(etiquetadas como `TODAS CONSOLIDADAS`). Si se cuentan juntas, se duplica la información.

In [ ]:
# Las filas agregadas están en las columnas de CLASE y CATEGORÍA,
# etiquetadas como "TODAS CONSOLIDADAS".
total_agregadas = 0
for nombre, d in crudos.items():
    cols_cat = [c for c in ['DESCRIPCIÓN CLASE', 'CLASE', 'DESCRIPCIÓN CATEGORÍA', 'CATEGORIA']
                if c in d.columns]
    mask = pd.Series(False, index=d.index)
    for col in cols_cat:
        mask = mask | d[col].astype(str).str.contains('TODA|CONSOLID', case=False, na=False)
    total_agregadas += mask.sum()

print(f'Filas agregadas ("TODAS CONSOLIDADAS") en total: {total_agregadas:,}')
print('Aparecen en las columnas de clase y categoría.')
print('Estas filas se ELIMINAN en la Fase 3 para quedarnos solo con el detalle.')

---
## 5 — Valores imposibles: TNOH mayor a 100%

La tasa de ocupación es un porcentaje: debería estar entre 0 y 100. Pero en los datos crudos
hay valores por encima de 100%, lo cual es físicamente imposible (no puedes ocupar más
habitaciones de las que existen).

In [ ]:
imposibles = []
for nombre, d in crudos.items():
    for col in ['TNOH EN EL MES(%)', 'PORCENTAJE_TNOH']:
        if col in d.columns:
            # convertimos a número (la coma decimal se vuelve punto)
            serie = pd.to_numeric(d[col].astype(str).str.replace(',', '.'), errors='coerce')
            malos = serie[serie > 100]
            if len(malos):
                imposibles.append({'archivo': nombre, 'casos_TNOH>100': len(malos),
                                   'valor_max': round(malos.max(), 2)})

imp = pd.DataFrame(imposibles)
if len(imp):
    print('Valores de TNOH imposibles (>100%) encontrados:')
    display(imp)
    print('\nEstos 👆 se eliminan en la Fase 3.')
else:
    print('No se encontraron valores imposibles.')

---
## 6 — Valores faltantes por columna

Revisamos el porcentaje de celdas vacías en el esquema viejo y en el nuevo, para saber qué
tan completos vienen los datos.

In [ ]:
def resumen_nulos(d, titulo):
    pct = (d.isna().mean() * 100).round(1)
    pct = pct[pct > 0].sort_values(ascending=False)
    print(f'=== {titulo} ===')
    if len(pct):
        print((pct.astype(str) + ' %').to_string())
    else:
        print('Sin valores nulos.')
    print()

resumen_nulos(viejo, 'Nulos en 2015 (esquema viejo)')
resumen_nulos(nuevo, 'Nulos en 2024 (esquema nuevo)')

---
## 7 — ¿Cómo se distribuye la ocupación?

Juntamos (solo para visualizar) el TNOH de todos los años y miramos su forma. Esto nos da la
primera intuición del comportamiento de la variable que queremos predecir.

In [ ]:
tnoh_all = []
for nombre, d in crudos.items():
    for col in ['TNOH EN EL MES(%)', 'PORCENTAJE_TNOH']:
        if col in d.columns:
            serie = pd.to_numeric(d[col].astype(str).str.replace(',', '.'), errors='coerce')
            # solo valores válidos, para la foto
            tnoh_all.append(serie[(serie >= 0) & (serie <= 100)])

tnoh = pd.concat(tnoh_all).dropna()
print(f'Observaciones válidas de TNOH: {len(tnoh):,}')
print(tnoh.describe().round(2).to_string())

plt.hist(tnoh, bins=60, color='#2E4057', alpha=0.85)
plt.axvline(tnoh.median(), color='#8B1A1A', ls='--', label=f'mediana = {tnoh.median():.1f}%')
plt.title('Distribución del TNOH (datos crudos, solo valores válidos)')
plt.xlabel('TNOH (%)'); plt.ylabel('frecuencia'); plt.legend()
plt.tight_layout(); plt.show()

**Observación:** la distribución está **sesgada a la derecha** — la mayoría de los
hoteles-mes ocupan poco (en torno al 25%) y pocos llegan a ocupaciones altas. Este sesgo es el
que justificará, en fases posteriores, usar el percentil 75 como umbral de "alta ocupación".

---
## 8 — Evolución de los arribos por año

Sumamos los arribos a hospedajes por año para ver la tendencia general — y de paso, el impacto
de la pandemia en 2020.

In [ ]:
por_anio = {}
for nombre, d in crudos.items():
    col_arr = next((c for c in ['TOTAL DE ARRIBOS EN EL MES', 'TOTAL_ARRIBOS'] if c in d.columns), None)
    cols_cat = [c for c in ['DESCRIPCIÓN CLASE', 'CLASE', 'DESCRIPCIÓN CATEGORÍA', 'CATEGORIA'] if c in d.columns]
    if col_arr and cols_cat:
        # quitamos las filas agregadas para no doblar el conteo
        agg = pd.Series(False, index=d.index)
        for col in cols_cat:
            agg = agg | d[col].astype(str).str.contains('TODA|CONSOLID', case=False, na=False)
        det = d[~agg]
        arr = pd.to_numeric(det[col_arr].astype(str).str.replace(',', '.'), errors='coerce')
        anio = nombre.replace('Indicadores_ocupabilidad_', '').replace('.csv', '')[:4]
        por_anio[anio] = por_anio.get(anio, 0) + arr.sum()

s = pd.Series(por_anio).sort_index()
plt.bar(s.index, s.values / 1e6, color='#C8A951')
plt.title('Arribos totales por año (millones) — datos crudos')
plt.ylabel('arribos (millones)'); plt.xlabel('año')
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()
print('Fíjate en la caída de 2020: el impacto del COVID.')
print('Ojo: 2025 solo trae el primer semestre, por eso su barra es menor.')
print((s/1e6).round(2).to_string())

---
## Resumen de la Fase 2 — lo que la Fase 3 debe resolver

La exploración dejó al descubierto **5 problemas** en los datos crudos. Ninguno se arregló
aquí; esta fase solo los **identificó**. La Fase 3 (Preparación) se encarga de resolverlos:

| # | Problema detectado | Cómo lo resuelve la Fase 3 |
|---|---|---|
| 1 | **Dos esquemas de columnas** incompatibles (2015–2023 vs 2024–2025) | Renombra todo a un esquema común antes de unir |
| 2 | **Separador decimal** distinto (coma vs. punto) | Unifica a punto al convertir a número |
| 3 | **Filas agregadas** ("TODAS CONSOLIDADAS") mezcladas con el detalle | Las elimina para quedarse solo con el detalle |
| 4 | **Valores imposibles** (TNOH > 100%) | Los descarta |
| 5 | **Distribución sesgada** del TNOH | Justifica binarizar en el percentil 75 |

Con el panorama claro, pasamos a la **Fase 3: Preparación de los datos**.